In [ ]:
# Continuación de tu "código final"...
# Asumimos que X_train, X_test, y_train, y_test ya existen y están preprocesados.
# y_train e y_test son DataFrames con las columnas objetivo que hayas definido
# (ej: ['ESCHOM', 'ESCMUJ', 'CIUOHOM', 'CIUOMUJ'])

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd # Para crear un DataFrame con los resultados
import numpy as np # Para np.nan en el DataFrame de resumen si fuera necesario

# Nombres de las variables objetivo para los reportes
# Asegúrate de que y_train sea un DataFrame para que .columns funcione
if isinstance(y_train, pd.DataFrame):
    target_names = y_train.columns.tolist()
else:
    print("Advertencia: y_train no es un DataFrame. Se usarán nombres de objetivo genéricos.")
    target_names = [f'Target_{i+1}' for i in range(y_train.shape[1])]


# --- Diccionario para almacenar los resultados del modelo ---
results_summary = {}

# --- Random Forest Regressor ---
print("\n--- Entrenando Modelo: Random Forest Regressor ---")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Evaluación para Random Forest Regressor
print("\nEvaluación - Random Forest Regressor:")
rf_metrics = {}
for i, target_name in enumerate(target_names):
    # Asegurarse de que y_test también se maneje correctamente si es DataFrame o array
    y_test_target_actual = y_test.iloc[:, i] if isinstance(y_test, pd.DataFrame) else y_test[:, i]
    y_pred_target_actual = y_pred_rf[:, i]

    mae = mean_absolute_error(y_test_target_actual, y_pred_target_actual)
    mse = mean_squared_error(y_test_target_actual, y_pred_target_actual)
    r2 = r2_score(y_test_target_actual, y_pred_target_actual)
    print(f"  Target: {target_name}")
    print(f"    Mean Absolute Error (MAE): {mae:.4f}")
    print(f"    Mean Squared Error (MSE): {mse:.4f}")
    print(f"    R-squared (R²): {r2:.4f}")
    rf_metrics[target_name] = {'MAE': mae, 'MSE': mse, 'R2': r2}
results_summary['Random Forest'] = rf_metrics

# --- Resumen de Resultados en un DataFrame ---
print("\n--- Resumen de Métricas para Random Forest ---")
summary_df_data = []
model_name = 'Random Forest' # Único modelo
metrics_dict = results_summary[model_name]

# Preparamos una fila para el DataFrame de resumen
row = {'Modelo': model_name}
for target_name in target_names:
    row[f'{target_name}_R2'] = metrics_dict.get(target_name, {}).get('R2', np.nan)
    row[f'{target_name}_MAE'] = metrics_dict.get(target_name, {}).get('MAE', np.nan)
    row[f'{target_name}_MSE'] = metrics_dict.get(target_name, {}).get('MSE', np.nan)
summary_df_data.append(row)

summary_df = pd.DataFrame(summary_df_data)
print(summary_df)
